# 1.Business Understanding

Given an online review, determine whether the sentiment is Positive, Negative, or Neutral.

# 2.Data Collection

In [1]:
import pandas as pd

data = {"review":["I REALLY loved this Phone!!! Best camera ever 😍😍 Visit https://bit.ly/x",
                    "Worst phone EVER!!! 😡😡 Battery drains in 2 hours... Don't buy!!!",
                    "Camera quality is AMAZING!!! 😊😊 Very happy with purchase. Visit www.shop.com",
                     "The phone is okay... Nothing special 🤔" ],
        "sentiment":["Positive",
                     "Negative",
                     "Positive",
                     "Neutral"]}

df = pd.DataFrame(data)
df

,review,sentiment
0,I REALLY loved this Phone!!! Best camera ever ...,Positive
1,Worst phone EVER!!! 😡😡 Battery drains in 2 hou...,Negative
2,Camera quality is AMAZING!!! 😊😊 Very happy wit...,Positive
3,The phone is okay... Nothing special 🤔,Neutral


# 3.Data Understanding

In [2]:
df.head()

,review,sentiment
0,I REALLY loved this Phone!!! Best camera ever ...,Positive
1,Worst phone EVER!!! 😡😡 Battery drains in 2 hou...,Negative
2,Camera quality is AMAZING!!! 😊😊 Very happy wit...,Positive
3,The phone is okay... Nothing special 🤔,Neutral


In [3]:
df.shape

(4, 2)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     4 non-null      object
 1   sentiment  4 non-null      object
dtypes: object(2)
memory usage: 192.0+ bytes


# 4.Data Preparation

In [ ]:
pip install nltk

In [ ]:
import nltk

nltk.download('stopwords')   # for removing common word
nltk.download('wordnet')     # Lemmatization
nltk.download('omw-1.4')     #Supporting Wordnet language resources

In [6]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [7]:
stop_words = set(stopwords.words("english"))  #This gets a list of common English stopwords from NLTK.
lemmatizer = WordNetLemmatizer()  # This creates a lemmatizer object from NLTK.The lemmatizer converts words into their base/dictionary form.

In [8]:
# Preprocessing functon

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "",text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.translate(str.maketrans('', '',string.punctuation))
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

df["clean_review"] = df["review"].apply(preprocess)
print(df[["review","clean_review"]])

                                              review  \
0  I REALLY loved this Phone!!! Best camera ever ...   
1  Worst phone EVER!!! 😡😡 Battery drains in 2 hou...   
2  Camera quality is AMAZING!!! 😊😊 Very happy wit...   
3             The phone is okay... Nothing special 🤔   

                                        clean_review  
0          really loved phone best camera ever visit  
1     worst phone ever battery drain 2 hour dont buy  
2  camera quality amazing happy purchase visit ww...  
3                         phone okay nothing special  


# 4.1 Bag Model

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
x_bow = cv.fit_transform(df["clean_review"])
bow_df = pd.DataFrame(x_bow.toarray(),columns=cv.get_feature_names_out())
bow_df

,amazing,battery,best,buy,camera,dont,drain,ever,happy,hour,...,nothing,okay,phone,purchase,quality,really,special,visit,worst,wwwshopcom
0,0,0,1,0,1,0,0,1,0,0,...,0,0,1,0,0,1,0,1,0,0
1,0,1,0,1,0,1,1,1,0,1,...,0,0,1,0,0,0,0,0,1,0
2,1,0,0,0,1,0,0,0,1,0,...,0,0,0,1,1,0,0,1,0,1
3,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,0,0,1,0,0,0


# 4.2 TF-IDF Vectorization

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(df["clean_review"])
tfidf_df = pd.DataFrame(X_tfidf.toarray(),columns=tfidf.get_feature_names_out())
tfidf_df.round(2)

,amazing,amazing happy,battery,battery drain,best,best camera,buy,camera,camera ever,camera quality,...,quality,quality amazing,really,really loved,special,visit,visit wwwshopcom,worst,worst phone,wwwshopcom
0,0.00,0.00,0.00,0.00,0.3,0.3,0.00,0.23,0.3,0.00,...,0.00,0.00,0.3,0.3,0.0,0.23,0.00,0.00,0.00,0.00
1,0.00,0.00,0.27,0.27,0.0,0.0,0.27,0.00,0.0,0.00,...,0.00,0.00,0.0,0.0,0.0,0.00,0.00,0.27,0.27,0.00
2,0.29,0.29,0.00,0.00,0.0,0.0,0.00,0.23,0.0,0.29,...,0.29,0.29,0.0,0.0,0.0,0.23,0.29,0.00,0.00,0.29
3,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.0,0.00,...,0.00,0.00,0.0,0.0,0.4,0.00,0.00,0.00,0.00,0.00


# 5.Model Building || 6.Model Training

In [11]:
from sklearn.linear_model import LogisticRegression
X= X_tfidf
y = df["sentiment"]
model = LogisticRegression()
model.fit(X,y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


# 7.Model Testing|8.Model Prediction

In [12]:
new_review = ["Good Camera and battery"]
new_review_clean = [preprocess(text) for text in new_review]
new_review_tfidf = tfidf.transform(new_review_clean)
prediction = model.predict(new_review_tfidf)
print(prediction)

['Positive']


# Applying Word2Vec

## 1.Install Word2Vec Library

In [ ]:
!pip install gensim

## 2.Importing Word2Vec

In [13]:
from gensim.models import Word2Vec
import numpy as np

## 3.Data Preparation

In [14]:
sentences =[review.split() for review in df["clean_review"]]
sentences

[['really', 'loved', 'phone', 'best', 'camera', 'ever', 'visit'],
 ['worst', 'phone', 'ever', 'battery', 'drain', '2', 'hour', 'dont', 'buy'],
 ['camera', 'quality', 'amazing', 'happy', 'purchase', 'visit', 'wwwshopcom'],
 ['phone', 'okay', 'nothing', 'special']]

### 3.1 Training the Word2Vec Model

In [17]:
word2vec_model = Word2Vec(sentences=sentences,vector_size=100,window=5,min_count=1,workers=4,sg=1)

word2vec_model

In [18]:
word2vec_model.wv["phone"]

array([-5.3622725e-04,  2.3643136e-04,  5.1033497e-03,  9.0092728e-03,
       -9.3029495e-03, -7.1168090e-03,  6.4588725e-03,  8.9729885e-03,
       -5.0154282e-03, -3.7633716e-03,  7.3805046e-03, -1.5334714e-03,
       -4.5366134e-03,  6.5540518e-03, -4.8601604e-03, -1.8160177e-03,
        2.8765798e-03,  9.9187379e-04, -8.2852151e-03, -9.4488179e-03,
        7.3117660e-03,  5.0702621e-03,  6.7576934e-03,  7.6286553e-04,
        6.3508903e-03, -3.4053659e-03, -9.4640139e-04,  5.7685734e-03,
       -7.5216377e-03, -3.9361035e-03, -7.5115822e-03, -9.3004224e-04,
        9.5381187e-03, -7.3191668e-03, -2.3337686e-03, -1.9377411e-03,
        8.0774371e-03, -5.9308959e-03,  4.5162440e-05, -4.7537340e-03,
       -9.6035507e-03,  5.0072931e-03, -8.7595852e-03, -4.3918253e-03,
       -3.5099984e-05, -2.9618145e-04, -7.6612402e-03,  9.6147433e-03,
        4.9820580e-03,  9.2331432e-03, -8.1579173e-03,  4.4957981e-03,
       -4.1370760e-03,  8.2453608e-04,  8.4986202e-03, -4.4621765e-03,
      

In [20]:
#Checking the similar words
word2vec_model.wv.most_similar("phone")

[('drain', 0.2188439965248108),
 ('happy', 0.21615555882453918),
 ('buy', 0.09311048686504364),
 ('wwwshopcom', 0.09291722625494003),
 ('loved', 0.08404714614152908),
 ('quality', 0.07964139431715012),
 ('really', 0.06437788903713226),
 ('amazing', 0.06283695995807648),
 ('2', 0.05433367192745209),
 ('purchase', 0.02706480585038662)]

## 3.2 Creating One vector for each review

In [26]:
def review_vector(review):
    words=review.split()

    word_vectors=[]

    for word in words:
        if word in word2vec_model.wv:
            word_vectors.append(word2vec_model.wv[word])

    if len(word_vectors)==0:
        return np.zeros(100)

    return np.mean(word_vectors,axis=0)

## 3.3 Converting all reviews into Word2Vec Vectors

In [27]:
X_word2vec = np.array([review_vector(review) for review  in df["clean_review"]])

X_word2vec

array([[-1.1108494e-03,  4.1669486e-03,  1.7939175e-03,  5.0822902e-03,
        -5.4089725e-04,  1.0162356e-03,  1.2805077e-03,  2.9042761e-03,
        -1.9860307e-04, -1.5174150e-03,  2.9412019e-03, -2.4230212e-04,
         3.7189948e-03,  1.8357991e-03,  1.9347767e-03, -1.8793595e-03,
         4.8274691e-03,  9.1992458e-04, -6.2403106e-03, -1.1510784e-03,
        -1.3682539e-04, -5.7112670e-04,  2.8514987e-04, -1.3549546e-04,
         5.4760062e-04,  4.3914065e-04,  7.1083481e-04,  1.6252795e-03,
        -1.4631439e-03,  1.9826954e-03,  1.5096038e-03, -5.1674345e-03,
         5.9080136e-04, -3.3107197e-03, -1.8429131e-03,  1.3930362e-03,
         2.3377431e-03, -9.3135273e-04, -1.7040556e-04,  6.5855467e-04,
        -4.7365722e-04,  1.9578878e-03, -1.9794551e-03,  1.9753794e-03,
        -3.5023206e-04, -7.4159885e-07, -8.3115959e-04, -1.9547825e-03,
        -4.3561126e-04,  1.9971260e-03,  1.2390589e-03, -1.9059574e-03,
        -5.7090558e-03, -2.6510786e-03, -2.1207121e-03,  6.13215

In [29]:
X_word2vec.shape

(4, 100)

In [30]:
word2vec_df = pd.DataFrame(X_word2vec)

word2vec_df.round(3)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.001,0.004,0.002,0.005,-0.001,0.001,0.001,0.003,-0.000,-0.002,...,-0.001,0.000,0.001,0.000,0.002,-0.000,-0.000,-0.000,-0.002,0.001
1,0.000,0.001,0.000,-0.001,0.002,-0.002,0.000,0.004,-0.001,-0.002,...,0.004,0.002,0.004,-0.000,0.002,0.003,0.001,0.001,0.001,0.003
2,-0.001,0.002,0.003,0.004,0.001,-0.004,0.001,0.003,-0.003,-0.005,...,-0.001,0.000,-0.001,0.002,0.004,-0.000,0.001,-0.001,0.001,-0.003
3,-0.002,-0.000,-0.001,-0.000,-0.004,-0.000,0.005,0.004,-0.005,-0.001,...,0.003,0.001,-0.001,-0.002,0.005,0.001,0.001,-0.004,0.001,0.002


## 4.Model Building | 5.Model Training

In [31]:
from sklearn.linear_model import LogisticRegression

X_w2v=X_word2vec
y=df["sentiment"]

word2vec_model_classifier = LogisticRegression()
word2vec_model_classifier.fit(X_w2v,y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## 6.Model Testing

In [33]:
new_review =["Good Camera and battery"]

new_review_clean = [preprocess(text) for text in new_review]

new_review_word2vec = np.array([review_vector(review) for review in new_review_clean])

prediction_word2vec=word2vec_model_classifier.predict(new_review_word2vec)

print(prediction_word2vec)

['Positive']
